# 1. Alpha-Beta Pruning

Aplică Alpha-Beta Pruning pe jocul X&0

In [2]:
import copy

class Game:
    MAX = 'X'
    MIN = '0'
    GOL = '.'

    def __init__(self, tabla=None):
        if tabla is None:
            self.tabla = [[Game.GOL for _ in range(3)] for _ in range(3)]
        else:
            self.tabla = copy.deepcopy(tabla)

    def jucator_opus(self, jucator):
        return Game.MIN if jucator == Game.MAX else Game.MAX

    def succesori(self, jucator):
        lista_mutari = []
        for i in range(3):
            for j in range(3):
                if self.tabla[i][j] == Game.GOL:
                    new_tabla = copy.deepcopy(self.tabla)
                    new_tabla[i][j] = jucator
                    lista_mutari.append(Game(new_tabla))
        return lista_mutari

    def linii_coloane_diag(self):
        linii = self.tabla
        coloane = [[self.tabla[r][c] for r in range(3)] for c in range(3)]
        diag_principala = [self.tabla[i][i] for i in range(3)]
        diag_secundara = [self.tabla[i][2 - i] for i in range(3)]
        return linii + coloane + [diag_principala, diag_secundara]

    def scop(self):
        combinatii = self.linii_coloane_diag()

        for linie in combinatii:
            if linie[0] != Game.GOL and linie[0] == linie[1] == linie[2]:
                return linie[0]

        for rand in self.tabla:
            if Game.GOL in rand:
                return False  # jocul continua

        return "remiza"


    def __repr__(self):
        return '\n'.join(['|'.join(rand) for rand in self.tabla])


class NodX0:
    def __init__(self, state, parinte=None):
        self.informatie = state
        self.parinte = parinte
        self.euristica = None
        self.succesori = []
        self.cel_mai_bun_succesor = None

    def scor_linie(self, linie):
        nr_x = linie.count(Game.MAX)
        nr_0 = linie.count(Game.MIN)

        if nr_x > 0 and nr_0 > 0:
            return 0

        punctaj = {0: 0, 1: 1, 2: 10, 3: 100}

        if nr_x > 0:
            return punctaj[nr_x]
        if nr_0 > 0:
            return -punctaj[nr_0]

        return 0

    def calculeaza_euristica(self):
        scor = 0
        for linie in self.informatie.linii_coloane_diag():
            scor += self.scor_linie(linie)

        self.euristica = scor
        return self.euristica

    def __str__(self):
        return str(self.informatie)

    def __repr__(self):
        return str(self)


class GrafX0:
    def __init__(self):
        self.nodStart = NodX0(Game())

    def succesori(self, nod_curent, jucator):
        stari_urmatoare = nod_curent.informatie.succesori(jucator)
        return [NodX0(stare, parinte=nod_curent) for stare in stari_urmatoare]

    def minmax(self, nod, adancime, is_max):
        if adancime == 0 or nod.informatie.scop() is not False:
            nod.calculeaza_euristica()
            return nod

        if is_max:
            nod.succesori = self.succesori(nod, Game.MAX)

            best_score = float('-inf')
            best_child = None

            for succesor in nod.succesori:
                self.minmax(succesor, adancime - 1, False)

                if succesor.euristica > best_score:
                    best_score = succesor.euristica
                    best_child = succesor

            nod.euristica = best_score
            nod.cel_mai_bun_succesor = best_child
            return nod

        else:
            nod.succesori = self.succesori(nod, Game.MIN)

            best_score = float('inf')
            best_child = None

            for succesor in nod.succesori:
                self.minmax(succesor, adancime - 1, True)

                if succesor.euristica < best_score:
                    best_score = succesor.euristica
                    best_child = succesor

            nod.euristica = best_score
            nod.cel_mai_bun_succesor = best_child
            return nod

    def alpha_beta(self, nod, adancime, alpha, beta, is_max):
        if adancime == 0 or nod.informatie.scop() is not False:
            nod.calculeaza_euristica()
            return nod.euristica

        if is_max:
            nod.succesori = self.succesori(nod, Game.MAX)
            best_score = float('-inf')

            for succesor in nod.succesori:
               best_score = max(best_score, self.alpha_beta(succesor, adancime - 1, alpha, beta, False))
               alpha = max(best_score, alpha)
               if alpha >= beta:
                   break

            nod.euristica = best_score
            return best_score

        else:
            nod.succesori = self.succesori(nod, Game.MIN)
            best_score = float('inf')

            for succesor in nod.succesori:
               best_score = min(best_score, self.alpha_beta(succesor, adancime - 1, alpha, beta, True))
               beta = min(best_score, beta)
               if alpha >= beta:
                   break

            nod.euristica = best_score
            return best_score



In [19]:
graf = GrafX0()
nod_v = graf.alpha_beta(graf.nodStart, 3, float('-INF'), float('INF'), True)
print(nod_v)

12


# 2. Blackjack

Implementati jocul Blackjack folosind algoritmul expectimax. Nu uitati ca la blackjack dealer-ul mereu da hit la 16 sau mai putin, si mereu stand daca are cel putin 17, deci nu aveti minimizer. Este, practic, purely player vs nature.

Se joaca cu "stand all 17s", i.e. daca dealer-ul are 17(contine un as care este evaluat la 11), trebuie sa dea stand(favorizeaza jucatorul putin).

upcard e cartea dealer-ului pe care o vedeti.

In [66]:
#ramane de implementat alg
#puteti hardcoda probabilitati sau cu random sampling(sau ce alte idei aveti voi). astea mi se par mie cele mai usoare de implementat.
import random
from functools import lru_cache

class Blackjack:
    PROBS = {2: 1/13, 3: 1/13, 4: 1/13, 5: 1/13, 6: 1/13,
             7: 1/13, 8: 1/13, 9: 1/13, 10: 4/13, 11: 1/13}

    def __init__(self, carti, upcard):
        self.carti = carti
        self.upcard = upcard

    def eval_hand(self, hand):
        suma = sum(hand)
        no_asi = hand.count(11)

        while suma > 21 and no_asi > 0:
            no_asi -= 1
            suma -= 10

        return suma
    def dealer(self, dealer_hand, jucator_suma):
        d_suma = self.eval_hand(dealer_hand)

        if d_suma > 21:
            return 1.0
        if d_suma >= 17:
            if d_suma > jucator_suma:
                return -1.0
            elif d_suma < jucator_suma:
                return 1.0
            else:
                return 0.0

        # hit
        ev = 0
        for carte, prob in self.PROBS.items():
            ev += prob * self.dealer(dealer_hand + [carte], jucator_suma)
        return ev


    def eval_stand(self):
        jucator_suma = self.eval_hand(self.carti)
        ev_total = 0

        for carte_ascunsa, prob in self.PROBS.items():
            ev_total += prob * self.dealer([self.upcard, carte_ascunsa], jucator_suma)

        return ev_total

    def expectimax(self, adancime):
        #trebuie adancime > 1 altfel o sa cam favorizeze stand oricand, mai ales cu probabilitati hardcodate. be careful
        #hit-ul vrem sa il evaluam recursiv.
        jucator_suma = self.eval_hand(self.carti)

        if jucator_suma > 21:
            return -1.0, "Bust"
        if adancime == 0:
            return self.eval_stand(), "Stand"

        ev_stand = self.eval_stand()
        ev_hit = 0
        for carte, prob in self.PROBS.items():
            child_node = Blackjack(self.carti + [carte], self.upcard)
            ev_child, _ = child_node.expectimax(adancime - 1)
            ev_hit += prob * ev_child

        if ev_hit > ev_stand:
            return ev_hit, "Hit"
        else:
            return ev_stand, "Stand"

    def play(self):
        def trage_carte():
            return random.choices(list(self.PROBS.keys()), weights=list(self.PROBS.values()))[0]

        while True:
            _, decizie = self.expectimax(adancime=3)
            if decizie == "Hit":
                self.carti.append(trage_carte())
                if self.eval_hand(self.carti) > 21:
                    print("Jucatorul a dat bust. Dealerul castiga!")
                    print(self.carti)
                    print(f"Suma: {sum(self.carti)}")
                    return
            else:
                break

        dealer_hand = [self.upcard, trage_carte()]
        while self.eval_hand(dealer_hand) < 17:
            dealer_hand.append(trage_carte())

        jucator_suma = self.eval_hand(self.carti)
        dealer_suma = self.eval_hand(dealer_hand)

        if dealer_suma > 21 or jucator_suma > dealer_suma:
            print("Jucatorul castiga!")
        elif jucator_suma < dealer_suma:
            print("Dealerul castiga!")
        else:
            print("Egalitate!")

        print(self.carti)


In [79]:
joc = Blackjack(carti=[10, 5], upcard=7)
joc.play()

Jucatorul a dat bust. Dealerul castiga!
[10, 5, 10]
Suma: 25


In [80]:
joc = Blackjack(carti=[10], upcard=1)
joc.play()

Dealerul castiga!
[10, 10]
